# Understanding the data first

In this section we show how it's important to understand how our data is structured and what can we get from it before we even start dealing with machine learning. The next part is on data sets preparation and data processing. The last part is about quality metrics as it's important to understand the output of our models.

## Static input data analysis

In this example we go through the Boston housing data set that is a well-known set of house prices in Boston. It has the following features:

* **crim**  per capita crime rate by town.
* **zn** proportion of residential land zoned for lots over 25,000 sq.ft.
* **indus** proportion of non-retail business acres per town.
* **chas** Charles River dummy variable (= 1 if tract bounds river; 0 otherwise).
* **nox** nitrogen oxides concentration (parts per 10 million).
* **rm** average number of rooms per dwelling.
* **age** proportion of owner-occupied units built prior to 1940.
* **dis** weighted mean of distances to five Boston employment centres.
* **rad** index of accessibility to radial highways.
* **tax** full-value property-tax rate per \$10000
* **ptratio** pupil-teacher ratio by town.
* **black** 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town.
* **lstat** lower status of the population (percent).
* **medv** median value of owner-occupied homes in \$1000s.

We can load the data with Pandas to check what's in the data set.

In [1]:
import pandas as pd
from sklearn.datasets import load_boston

# Loading the dataset with pandas

boston_data = load_boston(return_X_y=False)

boston_housing_df = pd.DataFrame(boston_data.data,columns=boston_data.feature_names)
#boston_housing_df.drop("ID", axis=1, inplace=True)
boston_housing_df.head()

ModuleNotFoundError: No module named 'sklearn'

The first step that you should do before going into any kind of preprocessing or training, is to check the profile of your data. Take a look how to profile your data with Pandas below.

In [ ]:
import pandas_profiling as pp

pp.ProfileReport(boston_housing_df)

## Dataset preparation

Quality measure are more about the input and output of the model. We can take the dataset and depending on the way how we divided it, we can measure the quality of our model. One of the common problem that each data scientist has is about how to divide the data set into training and testing data sets. To understand the following equations we need to introduce new designations. Let $L_{n}$ be our training data set of size $n$, $T_{m}$ our testing data set of size $m$, $M_{e}$ the number of misclassified cases, $I$ a function that return 1 if there is a match between predicted and label value and $e(d)$ the error rate of classifier $d$. We use also $X$ and $Y$ sets that we have already explained. We can write the error rate like following:
\begin{equation}
e(d)=\frac{M_{e}}{m}.
\end{equation}
It is the opposite to accuracy that is described later in this section. Error rate can be calculated differently depending on which method of data set preparation is used. There are few commonly used approached of how we can handle the training, testing and validation data sets:

- resubstitution -- R-method,
- hold-out -- H-method,
- cross-validation -- $\pi$-method,
- bootstrap,
- jackknife.

The first method is a very simple one. We have the same data set for training and testing. It is not the best solution if we consider to have a solid classifier. The error rate can be written as following:
\begin{equation}
e_{R}(d)=\frac{1}{n}\sum_{j=1}^{n}I(d(X_{j};L_{n})\neq Y_{j}).
\end{equation}
It means that we calculate the error rate for each element $j$ of our training data set and add 1 for each well predicted case. We need to divide it with $n$ which is the number of elements in the training data set. 

The second method is about dividing a data set into two data sets. It can be divided by half or other proportions. One set is our training data set and the second training data set. We can swap those sets and calculate the average of both sets. The error rate can be calculated as following:
\begin{equation}
e_{\tau}(\hat{d})=\frac{1}{m}\sum_{j=1}^{m}I(\hat{d}(X_{j}^{t};L_{n}\neq Y_{j}^{t}).
\end{equation}
Compared to resubstitution method it uses the testing data set only.
Cross-validation is the most common approach. It can be also called as rotation method. We need to divide the data set into $k$ subsets. The elements in each set are randomly chosen. One of those sets are taken as a testing set where the other sets are merged into a  training set. It should be repeated $k$ times for each $k$ subset. The error rate can be calculated like following:
\begin{equation}
e_{CV}(d)=\frac{1}{n}\sum_{j=1}^{n}I(\hat{d}(X_{j};L_{n}^{(-j)}\neq Y_{j}).
\end{equation}
A special case is when $k=m$. It means that we have subsets where each consist of just one element. This approach is known as leave-one-out or U-method.

Bootstrap method can be considered as an extension of resubstitution. The goal is to generate multiple sets from the main set by randomly selection. We use resubstitution method on each set and calculate an average error at the end:
\begin{equation}
e_{B}(d)=\frac{1}{B}\sum_{b=1}^{B}\frac{\sum_{j=1}^{n}I(Z_{j}\notin L_{n}^{\star b})I(d(X_{j};L^{\star b}_{n})\not Y_{j})}{\sum_{j=1}^{n}(Z_{j}\notin L^{\star b}_{n})}.
\end{equation}

## Data preprocessing

In this section we go through another popular data set - the Titanic data set. You can grab it from [https://www.kaggle.com/c/titanic/data](https://www.kaggle.com/c/titanic/data). The data consists of several features and the binary class of survival. 

In [ ]:
import pandas as pd

# Read the train dataset of Titanic dataset
titanic_df = pd.read_csv("../datasets/titanic-train.csv")
titanic_df.drop("PassengerId", axis=1, inplace=True)
titanic_df.head()

Some values are not avaialble (NaN), we cannot use such data to build a model. We need clean it up and drop three features that seems not to give any information. 

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES_TO_DROP = ["Name", "Ticket", "Cabin"]

# Fill missing values
titanic_df.fillna(titanic_df.mean(), inplace=True)

# Transform the dataset and perform one-hot encoding
titanic_dummies_df = pd.get_dummies(titanic_df.drop(FEATURES_TO_DROP, axis=1))

# Divide the dataset to have train and test sets
X_train, X_test, y_train, y_test = train_test_split(titanic_dummies_df.drop("Survived",                                                                             axis=1),
                                                    titanic_df["Survived"],
                                                    test_size=0.4)

First of all, we will create a basic logistic regression classifier, that will be a benchmark for the further analysis. 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

scaler = StandardScaler()

# Create a logistic regression model
lr_classifier = LogisticRegression()
lr_classifier.fit(scaler.fit_transform(X_train), 
                  y_train)

# Calculate the accuracy score
y_pred = lr_classifier.predict(scaler.transform(X_test))
accuracy_score(y_test, y_pred)

In [ ]:
# Display feature importance
pd.Series(lr_classifier.coef_[0], 
          index=X_test.keys())

As a next step we are going to include some more properties from the previously dropped textual columns. We will consider the following features:

* **Name**
* **Ticket**
* **Cabin**

Let's dive into the variables, for some better understanding.

In [ ]:
# Display some names
titanic_df["Name"].sample(10, random_state=43)

Some words seem to occur quite often. We can analyze the most frequent ones to see if they may enrich the data we already collected.

In [ ]:
# Split the names into pieces
titanic_df["Name"].str.lower()\
    .str.split()[:10]

In [ ]:
# Create a dataframe with a word as a single column
words_df = titanic_df["Name"].str.lower()\
    .str.split()\
    .apply(pd.Series)\
    .stack()\
    .reset_index(drop=True)\
    .to_frame(name="word")
words_df.head()

In [ ]:
# Display most frequent words descending
words_df.groupby("word")["word"].count()\
    .sort_values(ascending=False).head(n=10)

In [ ]:
SPECIAL_WORDS = ["mr.", "miss.", "mrs.", "master."]

# Define new columns by the presence of special words
for special_word in SPECIAL_WORDS:
    titanic_dummies_df["Name_%s" % special_word] = titanic_df["Name"].str.contains(
        special_word, case=False, regex=False).astype(int)
    
# Display some first rows
titanic_dummies_df.head()

From the **Ticket** column we can also try to extract some more properties.

In [ ]:
# Show 10 randomly selected rows
titanic_df["Ticket"].sample(10, random_state=214)

In [ ]:
# Extract all the names from the 
titanic_df["Ticket"].str.findall(r"[A-Za-z]+")\
    .head(n=10)

In [ ]:
# Create a dataframe with a word as a single column
words_df = titanic_df["Ticket"].str.findall(r"[A-Za-z]+")\
    .apply(pd.Series)\
    .stack()\
    .reset_index(drop=True)\
    .to_frame(name="word")
words_df.head()

In [ ]:
# Convert the presence of the phrases to features
for ticket_word in words_df["word"].unique():
    titanic_dummies_df["Ticket_%s" % ticket_word] = titanic_df["Ticket"].str.contains(
        ticket_word, case=False, regex=False).astype(int)
    
# Display some first rows
titanic_dummies_df.head()

And the last column is *Cabin* where we could expect some similar process to be taken.

In [ ]:
# Filter out NaNs and show 10 randomly selected samples
not_null_mask = titanic_df["Cabin"].notna()
titanic_df[not_null_mask]["Cabin"].sample(20, random_state=134)

In the example of *Cabin* the letter may indicate closeness of different rooms. We may keep just this information - it won't tell us anything about an exact location of the cabins, but just a rough estimate of it.

In [ ]:
# Get all the letters that occur in the cabin number
letters_df = titanic_df[not_null_mask]["Cabin"].str.findall(r"[A-Za-z]")\
    .apply(pd.Series)\
    .stack()\
    .reset_index(drop=True)\
    .to_frame(name="letter")
# Display unique values
letters_df["letter"].unique()

In [ ]:
# Convert the presence of the letter to features
for letter in letters_df["letter"].unique():
    titanic_dummies_df["Cabin_%s" % letter] = titanic_df["Cabin"].str.contains(
        letter, case=False, regex=False, na=False).astype(int)
    
# Display some first rows
titanic_dummies_df.head()

### Logistic regression with engineered features

We have created another model, which includes some more features than the original one. We can directly include them for the model training, so we can then compare the achieved accuracy.

In [ ]:
# Replace the datasets with their extensions
X_train = titanic_dummies_df.iloc[X_train.index].drop("Survived", 
                                                      axis=1)
X_test = titanic_dummies_df.iloc[X_test.index].drop("Survived", 
                                                    axis=1)

In [ ]:
scaler = StandardScaler()

# Create a logistic regression model
lr_classifier = LogisticRegression()
lr_classifier.fit(scaler.fit_transform(X_train), 
                  y_train)

# Calculate the accuracy score
y_pred = lr_classifier.predict(scaler.transform(X_test))
accuracy_score(y_test, y_pred)

In [ ]:
# Display feature importance
pd.Series(lr_classifier.coef_[0], 
          index=X_test.keys())

## Model performance

The model performance can be measure in different ways. On of such measure is the loss function measurement. We have different methods to do it.

### Loss function

It is a method of evaluating how well your algorithm models your dataset. If your predictions are totally off, your loss function will output a higher number. If they’re pretty good, it’ll output a lower one.

Examples:

**Mean Squared Error** (MSE) - the workhorse of basic loss functions: it’s easy to understand and implement, and generally works pretty well. To calculate MSE, you take the difference between your predictions and the ground truth, square it, and average it out across the whole dataset. Often used in regression.

$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_{i}^{true} - y_{i}^{pred})^2$

**Cross Entropy** (log loss) - measures the performance of a classification model whose output is a probability value between 0 and 1. Cross-entropy loss increases as the predicted probability diverges from the actual label.

$H = -\frac{1}{n} \sum_{i=1}^{n} [y_{i}^{true} \log(y_{i}^{pred}) + (1 - y_{i}^{true}) \log(1 - y_{i}^{pred})]$

### Gradient based optimization

Gradient descent is a first-order iterative optimization algorithm for finding the minimum of a function. To find a local minimum of a function using gradient descent, one takes steps proportional to the negative of the gradient (or approximate gradient) of the function at the current point. If, instead, one takes steps proportional to the positive of the gradient, one approaches a local maximum of that function; the procedure is then known as gradient ascent.

![Minimum and maximum of the function](../images/minmax.png)

Vanilla gradient descent (batch gradient descent) computes the gradient of the cost function with respect to the parameters for the entire training dataset:

$\theta = \theta - \eta \cdot \nabla_\theta J(\theta)$

where:

- $\theta$ - parameters
- $\eta$ - learning rate
- $J$ - cost function


**Stochastic Gradient Descent** (SGD)

Stochastic gradient descent in contrast performs a parameter update for each training example:

$\theta = \theta - \eta \cdot \nabla_\theta J( \theta, x_i, y_i)$

where:

- $x_i$ - example
- $y_i$ - label

Batch gradient descent performs redundant computations for large datasets, as it recomputes gradients for similar examples before each parameter update. SGD does away with this redundancy by performing one update at a time. It is therefore usually much faster and can also be used to learn online.

**Mini-batch gradient descent**

Mini-batch gradient descent finally takes the best of both worlds and performs an update for every mini-batch of N training examples:

$\theta = \theta - \eta \cdot \nabla_\theta J( \theta, x_{(i:i + N)}, y_{(i:i + N)})$


Mini-batch approach reduces the variance of the parameter updates (more stable convergence) and make use of highly optimized matrix operations. It is typically the algorithm of choice when training a neural network and the term **SGD** usually is employed also when mini-batches are used.

**SGD optimizations**

There are many modifications to standard SGD method that improve robustness, reduce oscillation and gain faster convergence.

**Other**

There are plenty of optimizers like **Momentum**, **Nesterov Accelerated Gradient** (NAG), **Adagrad**, **Adadelta**, **RMSprop**, or **Adam**. Take a look at a comparison.

Gradient based method visualization |

![SGD variants summary](../images/gradsummary.gif) 
![SGD variants summary](../images/gradsaddlesummary.gif)

Images source: https://www.semanticscholar.org/paper/An-overview-of-gradient-descent-optimization-Ruder/769ef3d5021cd71c37d2c403f231a53d1accf786

Let's take a look how to use SGD optimizer with Keras. It can be done in a similar way in Tensorflow.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import SGD
from keras.utils.np_utils import to_categorical
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

features, labels = load_iris(return_X_y=True)
features = normalize(features)
labels = to_categorical(labels)

x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.15, shuffle=True)

model = Sequential()
model.add(Dense(units=10, activation='relu', input_dim=x_train.shape[1]))
model.add(Dense(units=7, activation='relu'))
model.add(Dense(units=3, activation='softmax'))

optimizer = SGD(lr=0.05, momentum=0.7)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.fit(x_train, y_train, epochs=150, batch_size=16, verbose=0)
loss, accuracy = model.evaluate(x_test, y_test)

print('accuracy on test set: {accuracy * 100}%')

We can apply different optimizers or loss functions to the same or different classification problems. It does not depends on the neural network architecture. A more complex example using MNIST dataset is shown below.

In [ ]:
import numpy as np
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPool2D
from keras.optimizers import Adam
from keras.utils.np_utils import to_categorical


FEATURES_SHAPE = (-1, 28, 28, 1)
MAX_FEATURE = 255

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = (x_train / MAX_FEATURE).astype(np.float16).reshape(FEATURES_SHAPE)
y_train = to_categorical(y_train)
x_test = (x_test / MAX_FEATURE).astype(np.float16).reshape(FEATURES_SHAPE)
y_test = to_categorical(y_test)

model = Sequential()
model.add(Conv2D(filters=6, kernel_size=5, activation='relu', input_shape=FEATURES_SHAPE[1:]))
model.add(MaxPool2D())
model.add(Conv2D(filters=16, kernel_size=3, activation='relu'))
model.add(MaxPool2D())
model.add(Flatten())
model.add(Dense(units=120, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(units=84, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(units=10, activation='softmax'))

optimizer = Adam(lr=0.005)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.fit(x_train, y_train, epochs=10, batch_size=128, verbose=1)
loss, accuracy = model.evaluate(x_test, y_test)

print('accuracy on test set: {accuracy * 100}%')

## Output quality metrics

There are several metrics to show the quality of our classification model:

- ROC that stands for Receiver Operating Characteristic curve,
- AUC -- Area Under Curve,
- $F_{1}$ score,
- Precision,
- Recall.

To calculate the metrics we ned 

|                      |condition positive |condition negative |
|----------------------|-------------------|-------------------|
|**predicted positive**|True Positive (TP) |False Positive (FP)|         
|**predicted negative**|False Negative (FN)|True Negative (TN) |

Most common metric is the accuracy. It can be calculated like following:

\begin{equation}
ACC=\frac{\#TP+\#TN}{\#TP+\#TN+\#FP+\#FN}.
\end{equation}

First one that we describe is called True Positive Rate (TPR). It can be calculated like following:
\begin{equation}
TPR=\frac{\#TP}{\#TP+\#FN}.
\end{equation}
TPR is also called sensitivity or recall and is a measure of good predictions within a set of cases. By $\#TP, \#FP$ we mean the number of True Positive and False Positive cases. An opposite to it is specificity. It is also called TNR what stands for True Negative Rate. It can be calculated as following:
\begin{equation}
TNR=\frac{\#TN}{\#TN+\#FP}.
\end{equation}
It is a measure that says how good we are at predicting negative scenario. Another important metric is precision that is also known as Positive Predictive Value (PPV):
\begin{equation}
PPV=\frac{\#TP}{\#TP+\#FP}.
\end{equation}
It is a ratio of positive cases that that were well predicted to all positive cases, even those that are not well predicted. The opposite to it is the Negative Predictive Value:
\begin{equation}
NPV=\frac{TN}{TN+FN}.
\end{equation}
We can also calculate the False Positive Rate metric known as fall-out. It is about how bad we are on predicting positive cases:
\begin{equation}
FPR=1-TNR.
\end{equation}
The opposite to FPR is False Negative Rate:
\begin{equation}
FNR=1-TPR.
\end{equation}
Another popular metric is called $F_{1}$ score and it is a weighted accuracy measure. It takes PPV and TPR to calculate the score:
\begin{equation}
F_{1}=2\frac{PPV\cdot TPR}{TPR+PPV}.
\end{equation}
The $F_{1}$ value as in case of all previous metrics between 1 and 0, where 1 is the best. 
A interesting measure is Matthews Correlation Coefficient measure that is about the correlation between observed and predicted values. The value of MCC is between -1 and 1. If we have a perfect classifier we get MCC=1. A random classifier is when we have MCC=0 and a totally bad classifier if have MCC=-1. This measure can be calculated as following:
\begin{equation}
MCC=\frac{\#TP\cdot\#TN-\#FP\cdot\#FN}{\sqrt{(\#TP+\#FP)(\#TP+\#FN)(\#TN+\#FP)(\#TN+\#FN)}}.
\end{equation}

Let's generate a test and train dataset to measure the metrics.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler

# build the train dataset
X, y = make_blobs(n_samples=1000, centers=2, n_features=2, random_state=1)
scalar = MinMaxScaler()
scalar.fit(X)
X = scalar.transform(X)

# build the test dataset
Xtest, ytest = make_blobs(n_samples=5000, centers=2, n_features=2, random_state=1)
Xtest = scalar.transform(Xtest)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()
model.add(Dense(4, input_dim=2, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# compile it with adam optimizer
model.compile(loss='binary_crossentropy', optimizer='adam')
model.fit(X, y, epochs=10, verbose=0)

ypred = model.predict_classes(Xtest)

In [ ]:
print(len(ypred))

In [ ]:
import math

def calculate_quality_metrics(y,ypredicted):
    tn = 0
    tp = 0
    fn = 0
    fp = 0
    for i in range(len(y)):
        if y[i] > 0:
            if y[i] == ypredicted[i]:
                tp = tp + 1
            else:
                fp = fp + 1
        else:
            if y[i] == ypredicted[i]:
                tn = tn + 1
            else:
                fn = fn + 1
    acc = ((tp + tn) * 1.0) / ((tp + tn + fp + fn) * 1.0)
    tpr = tp * 1.0 / (tp + fn) * 1.0
    tnr = tn * 1.0 / (tn + fp) * 1.0
    ppv = tp / (tp + fp) * 1.0
    npv = tn / (tn + fn) * 1.0
    fpr = 1.0 - tnr
    fnr = 1.0 - tpr
    f1 = 2 * (ppv * tpr * 1.0 / (tpr + ppv * 1.0))
    mcc = (tp * tn - fp * fn) / (math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) * 1.0)    
    print("Accuracy: "+str(acc))
    print("TPR: "+str(tpr))
    print("TNR: "+str(tnr))
    print("PPC: "+str(ppv))
    print("NPV: "+str(npv))
    print("FPR: "+str(fpr))
    print("FNR: "+str(fnr))
    print("MCC: "+str(mcc))
    print("F1: "+str(f1))    
    return [acc, tpr, tnr, ppv, npv, fpr, fnr, mcc, f1]

In [ ]:
calculate_quality_metrics(ytest,ypred)